<a href="https://colab.research.google.com/github/mryaryyev/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installs

In [21]:
!pip install -q transformers torch bitsandbytes datasets accelerate peft trl langdetect

## Imports

In [22]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

## Tokenizer & Quantization Config

In [23]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print("Vocab size:", tokenizer.vocab_size)
# print("Special Tokens:", tokenizer.special_tokens_map)

# quantization => yukaridaki base modelnin hacmini kucultup yuklenmesi (4k => 1080p)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

## Load Base Model (4-bit)

In [24]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto", # GPU, CPU
    dtype=torch.float16,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Before Fine-tuning (Baseline Output)

In [25]:
prompt = "Explain what a tokenizer is?"
# prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        max_length=None,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Explain what a tokenizer is?


## Inspect Model Architecture

In [26]:
# print(model)
firs_block = model.model.layers[0]
print(firs_block)
print("=========")
print(firs_block.self_attn)
print("=========")
print(model.config)

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (o_proj): Linear4b

## Count Parameters

In [27]:
def count_parameters(model): #parameters: attentiondaki q,k,v projections, weight, bias, embedding matrix, attention matrix
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load_dataset
* instruction tuning

In [28]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset
dataset[1]

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

## Prompt Formatting

In [29]:
def generate_prompt(example):
  instruction = example["instruction"]
  input_text = example["input"]
  output_text = example["output"]

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )

# print(generate_prompt(dataset[0]))

def formatting_func(example):
  return {"text": generate_prompt(example)}

dataset = dataset.map(formatting_func)

## Check Formatted Sample

In [30]:
print(dataset[0]["text"])

### Instruction:
Give three tips for staying healthy.

### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.




## Shuffle & Select Dataset

In [31]:
dataset = dataset.shuffle(seed=42) # seed=42 => yarin calistirsanda ayni hizada datalar olur

In [32]:
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1] => exact same order

In [33]:
dataset = dataset.select(range(7000))

In [34]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

## Fine-tuning Terms (Notes)

In [35]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter-Efficient Fine Tuning
# OOM => Out of Memory

## LoRA Config

In [36]:
lora_config = LoraConfig(
    r=8, # rank => hacim
    lora_alpha=16, # scaling amalga oshiradi
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

## Apply LoRA

In [37]:
model = get_peft_model(model, lora_config)

In [38]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


## QLoRA Notes

In [40]:
# QLoRA
# LoRa

## Training Args

In [23]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # VRAM
    num_train_epochs=1, # kop bolsa overfit bolishi mumkun
    logging_steps=20,
    output_dir="./jaydari_gpt",
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    report_to="none"
)

## Trainer

In [24]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x["text"],
    args=training_args
)

Applying formatting function to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

## Fix Dtype

In [25]:
print("before:", {p.dtype for p in trainer.model.parameters() if p.requires_grad})

for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

print("after:", {p.dtype for p in trainer.model.parameters() if p.requires_grad})

before: {torch.bfloat16}
after: {torch.float32}


## Train

In [27]:
trainer.train()
model.save_pretrained("jaydari_gpt")
tokenizer.save_pretrained("jaydari_gpt")

Step,Training Loss
20,1.569456
40,1.475851
60,1.385442
80,1.357623
100,1.241497
120,1.292830
140,1.274282
160,1.228221
180,1.228989
200,1.293645


('jaydari_gpt/tokenizer_config.json',
 'jaydari_gpt/chat_template.jinja',
 'jaydari_gpt/tokenizer.json')

## Base Model (fp16)

In [41]:
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)

model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    dtype=torch.float16
)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

## Base Output

In [42]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        max_length=None,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

===== BASE MODEL OUTPUT =====
Explain what machine learning is in simple words.

- What are some of the applications of machine learning in the software industry?


#FINE-TUNED MODEL INFERENCE

In [43]:
model_path = "jaydari_gpt"

tokenizer_ft = AutoTokenizer.from_pretrained(model_path)

model_ft = PeftModel.from_pretrained(
    model_base,
    model_path
)

model_ft.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

## Fine-Tuned Output

In [44]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs_ft = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device)

with torch.no_grad():
    output_ft = model_ft.generate(
        **inputs_ft,
        max_new_tokens=120,
        max_length=None,
        temperature=0.7,
        do_sample=True
    )

print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer_ft.decode(output_ft[0], skip_special_tokens=True))

===== FINE-TUNED MODEL OUTPUT =====
### Instruction:
Explain what machine learning is in simple words.

### Response:
Machine learning is an area of artificial intelligence that involves the use of algorithms to learn from data without any human intervention. The use of machine learning can help computers perform tasks that humans can do, such as analyzing vast amounts of data, making predictions, and identifying patterns. The algorithms behind machine learning are trained on vast amounts of data, allowing computers to take over tasks that would otherwise be too complex or time-consuming for humans. This allows computers to learn from experience and improve their performance over time.




## Backup

In [50]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r jaydari_gpt /content/drive/MyDrive/

Mounted at /content/drive


In [51]:
!ls -lh /content/drive/MyDrive/jaydari_gpt

total 7.8M
-rw------- 1 root root 1.1K Nov  2 02:05 adapter_config.json
-rw------- 1 root root 4.4M Nov  2 02:05 adapter_model.safetensors
-rw------- 1 root root  410 Nov  2 02:05 chat_template.jinja
drwx------ 2 root root 4.0K Nov  2 02:05 checkpoint-1750
-rw------- 1 root root 1.5K Nov  2 02:05 README.md
-rw------- 1 root root  398 Nov  2 02:05 tokenizer_config.json
-rw------- 1 root root 3.5M Nov  2 02:05 tokenizer.json


# Model Testing

## Ask Function (Language Check + Inference)

In [45]:
model = model_ft
tokenizer = tokenizer_ft

In [46]:
from langdetect import detect_langs

def ask_jaydari_gpt(prompt: str, max_new_tokens=256):
  # Detect the language of the question
    try:
        langs = detect_langs(prompt)
        lang_code = langs[0].lang
        confidence = langs[0].prob
        print(f"🌍 Detected lang: {lang_code} ({confidence:.2f})")
    except:
        lang_code = "unknown"
        confidence = 0.0

    # Block only if we're confident it's NOT English
    if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:
        print("Sorry, you should ask a question only in English.")
        return

    # Build the prompt in the same format used for training
    full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        max_length=None,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    # Keep only the newly generated tokens (the answer)
    answer_ids = outputs[0][inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(answer_ids, skip_special_tokens=True).strip())

## Test English

In [ ]:
ask_jaydari_gpt("Can you tell about Python?")

## Test Non-English

In [9]:
ask_jaydari_gpt("Python hakkında bilgi verebilir misiniz??")

🌍 Detected lang: tr (1.00)
Sorry, you should ask a question only in English.


## Restore

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/jaydari_gpt /content/
!ls -lh jaydari_gpt